# Modul 12: Unüberwachtes Lernen und Textklassifikation

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Unüberwacht lernen, Text als Merkmale  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschritten  
    **Orientierungszeit:** etwa 130 bis 180 Minuten

    ## Überblick

    Sie vergleichen mehrere Cluster- und Anomalieverfahren, nutzen wenige Labels semi-supervised und bauen anschließend vollständig lokale Textmerkmale und Textklassifikationspipelines auf.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_12A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_12B_20260823.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - KMeans, MiniBatchKMeans, hierarchisches Clustering, DBSCAN, OPTICS und GaussianMixture vergleichen.
- IsolationForest, LocalOutlierFactor und OneClassSVM auf skalierten Daten anwenden.
- Semi-supervised Verfahren mit wenigen Labels untersuchen und Unsicherheit dokumentieren.
- Kleine lokale Textsammlungen mit Labels strukturieren und bereinigen.
- Count-, TF-IDF- und Hashing-Vektorisierung erzeugen.
- Einfache Textklassifikationspipelines trainieren, validieren, analysieren und speichern.

    ## Bewertete Fähigkeiten

    - Clusterlabels, Rauschen, Silhouette und Modellannahmen
- Anomaliescores und Novelty Detection
- LabelPropagation und LabelSpreading mit maskierten Labels
- CountVectorizer, TfidfVectorizer, HashingVectorizer und Sparse-Matrizen
- MultinomialNB, LogisticRegression, Fehleranalyse und Serialisierung

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import io
import joblib
import re
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans, MiniBatchKMeans, OPTICS
from sklearn.datasets import make_blobs, make_moons
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.svm import OneClassSVM
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

X_cluster_12, hidden_cluster_12 = make_blobs(
    n_samples=330,
    centers=[(-4, -2), (0, 4), (4, -1)],
    cluster_std=[0.55, 1.35, 0.75],
    random_state=RANDOM_SEED,
)
X_cluster_12 = np.vstack(
    [X_cluster_12, np.array([[9, 9], [-9, 7], [8, -8], [-8, -7]], dtype=float)]
)

X_ssl_12, y_ssl_12 = make_moons(
    n_samples=300,
    noise=0.12,
    random_state=RANDOM_SEED,
)

text_samples_12 = [
    ("Die Rechnung enthält einen falschen Betrag", "abrechnung"),
    ("Meine Kreditkarte wurde doppelt belastet", "abrechnung"),
    ("Bitte erklären Sie die monatliche Gebühr", "abrechnung"),
    ("Ich brauche eine korrigierte Rechnung", "abrechnung"),
    ("Die Erstattung ist noch nicht angekommen", "abrechnung"),
    ("Der Rabatt fehlt auf meiner Rechnung", "abrechnung"),
    ("Mein Login funktioniert seit heute nicht", "technik"),
    ("Die App stürzt beim Start sofort ab", "technik"),
    ("Ich kann mein Passwort nicht zurücksetzen", "technik"),
    ("Die Verbindung zum Server bricht ab", "technik"),
    ("Nach dem Update bleibt der Bildschirm leer", "technik"),
    ("Der Download endet immer mit einem Fehler", "technik"),
    ("Wie kann ich mein Paket verfolgen", "versand"),
    ("Die Lieferung ist mehrere Tage verspätet", "versand"),
    ("Das Paket wurde an die falsche Adresse geschickt", "versand"),
    ("Wann wird meine Bestellung versendet", "versand"),
    ("Der Zustellstatus hat sich nicht verändert", "versand"),
    ("Ich möchte den Liefertermin ändern", "versand"),
    ("Bitte ändern Sie meine hinterlegte E-Mail-Adresse", "konto"),
    ("Ich möchte mein Kundenkonto schließen", "konto"),
    ("Wo kann ich meine Profildaten bearbeiten", "konto"),
    ("Meine Telefonnummer im Konto ist falsch", "konto"),
    ("Ich brauche eine Kopie meiner gespeicherten Daten", "konto"),
    ("Wie aktiviere ich die Zwei-Faktor-Anmeldung", "konto"),
]
text_data_12 = pd.DataFrame(text_samples_12, columns=["text", "label"])

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Clusterverfahren und Silhouette vergleichen

    Skalieren Sie `X_cluster_12` und vergleichen Sie:

- KMeans,
- MiniBatchKMeans,
- AgglomerativeClustering,
- DBSCAN,
- OPTICS,
- GaussianMixture.

Berechnen Sie Anzahl gefundener Cluster, Anteil als Rauschen markierter Punkte und, sofern möglich, Silhouette Score. Visualisieren Sie jede Zuordnung in einer eigenen Abbildung.

> **Hinweis:** Entfernen Sie Rauschlabel -1 nur für die Silhouette-Berechnung, nicht aus der Dokumentation.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Clusterverfahren und Silhouette vergleichen
#
# Ziel dieser Codezelle:
# Skalieren Sie Xcluster12 und vergleichen Sie: - KMeans, - MiniBatchKMeans, -
# AgglomerativeClustering, - DBSCAN, - OPTICS, - GaussianMixture. Berechnen Sie
# Anzahl gefundener Cluster, Anteil als Rauschen markierter Punk...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster_12)

clusterers = {
    "KMeans": KMeans(n_clusters=3, n_init=10, random_state=RANDOM_SEED),
    "MiniBatchKMeans": MiniBatchKMeans(n_clusters=3, n_init=10, batch_size=64, random_state=RANDOM_SEED),
    "Agglomerativ": AgglomerativeClustering(n_clusters=3),
    "DBSCAN": DBSCAN(eps=0.28, min_samples=7),
    "OPTICS": OPTICS(min_samples=7, xi=0.05, min_cluster_size=0.08),
    "GaussianMixture": GaussianMixture(n_components=3, random_state=RANDOM_SEED),
}

clustering_rows = []
clustering_labels = {}

for name, model in clusterers.items():
    if isinstance(model, GaussianMixture):
        labels = model.fit_predict(X_scaled)
    else:
        labels = model.fit_predict(X_scaled)

    clustering_labels[name] = labels
    non_noise_mask = labels != -1
    unique_non_noise = np.unique(labels[non_noise_mask])
    cluster_count = len(unique_non_noise)
    noise_share = float(np.mean(labels == -1))

    # Silhouette benötigt mindestens zwei Cluster und mehr Punkte als Cluster.
    if cluster_count >= 2 and non_noise_mask.sum() > cluster_count:
        score = silhouette_score(
            X_scaled[non_noise_mask],
            labels[non_noise_mask],
        )
    else:
        score = np.nan

    clustering_rows.append(
        {
            "method": name,
            "clusters": cluster_count,
            "noise_share": noise_share,
            "silhouette": score,
        }
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, alpha=0.75)
    ax.set_title(name)
    ax.set_xlabel("Skaliertes Merkmal 1")
    ax.set_ylabel("Skaliertes Merkmal 2")
    plt.tight_layout()
    plt.show()

clustering_comparison = pd.DataFrame(clustering_rows).sort_values(
    "silhouette",
    ascending=False,
    na_position="last",
)
print(clustering_comparison.round(3).to_string(index=False))

### Reflexion zu Aufgabe 1

Die Verfahren optimieren unterschiedliche Vorstellungen von Gruppen. KMeans und GaussianMixture benötigen eine vorgegebene Clusterzahl. DBSCAN und OPTICS können Rauschen markieren und nicht-kugelförmige Gruppen finden, reagieren aber auf Dichteparameter. Silhouette bewertet Kompaktheit und Trennung im gewählten Merkmalsraum, ersetzt jedoch keine fachliche Plausibilitätsprüfung.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Anomalieverfahren auf skalierten Daten vergleichen

    Verwenden Sie die skalierten Clusterdaten und vergleichen Sie:

- `IsolationForest`,
- `LocalOutlierFactor`,
- `OneClassSVM`.

Markieren Sie jeweils ungefähr vier Prozent der Beobachtungen als ungewöhnlich, soweit das Verfahren dies erlaubt. Erstellen Sie eine Tabelle mit Anomalielabeln und Scores, zählen Sie Übereinstimmungen und visualisieren Sie Punkte, die von mindestens zwei Verfahren markiert werden.

> **Hinweis:** Vergleichen Sie zunächst Labels oder Ränge, nicht rohe Scores unterschiedlicher Modelle.

In [ ]:
# Nutzen Sie `X_scaled` aus Aufgabe 1 oder skalieren Sie erneut.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Anomalieverfahren auf skalierten Daten vergleichen
#
# Ziel dieser Codezelle:
# Verwenden Sie die skalierten Clusterdaten und vergleichen Sie: - IsolationForest,
# - LocalOutlierFactor, - OneClassSVM. Markieren Sie jeweils ungefähr vier Prozent
# der Beobachtungen als ungewöhnlich, soweit das Verfahr...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

if "X_scaled" not in globals():
    X_scaled = StandardScaler().fit_transform(X_cluster_12)

isolation_forest = IsolationForest(
    contamination=0.04,
    random_state=RANDOM_SEED,
)
isolation_labels = isolation_forest.fit_predict(X_scaled)
isolation_score = -isolation_forest.score_samples(X_scaled)

local_outlier_factor = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.04,
)
lof_labels = local_outlier_factor.fit_predict(X_scaled)
lof_score = -local_outlier_factor.negative_outlier_factor_

one_class_svm = OneClassSVM(nu=0.04, gamma="scale")
svm_labels = one_class_svm.fit_predict(X_scaled)
svm_score = -one_class_svm.decision_function(X_scaled).ravel()

anomaly_table = pd.DataFrame(
    {
        "isolation_anomaly": isolation_labels == -1,
        "isolation_score": isolation_score,
        "lof_anomaly": lof_labels == -1,
        "lof_score": lof_score,
        "svm_anomaly": svm_labels == -1,
        "svm_score": svm_score,
    }
)
anomaly_table["vote_count"] = anomaly_table[
    ["isolation_anomaly", "lof_anomaly", "svm_anomaly"]
].sum(axis=1)
consensus_mask = anomaly_table["vote_count"] >= 2

print("Anomalien je Verfahren:")
print(
    anomaly_table[
        ["isolation_anomaly", "lof_anomaly", "svm_anomaly"]
    ].sum().to_string()
)
print("Von mindestens zwei Verfahren markiert:", int(consensus_mask.sum()))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_scaled[:, 0], X_scaled[:, 1], alpha=0.5, label="übrige Punkte")
ax.scatter(
    X_scaled[consensus_mask, 0],
    X_scaled[consensus_mask, 1],
    facecolors="none",
    edgecolors="black",
    s=160,
    label="mindestens zwei Stimmen",
)
ax.set_title("Konsens-Anomaliehinweise")
ax.legend()
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 2

Die Scores verschiedener Verfahren besitzen unterschiedliche Skalen und dürfen nicht direkt numerisch verglichen werden. Ein Konsens kann die Priorisierung erleichtern, beweist aber weiterhin keine fehlerhafte Beobachtung. LOF bewertet lokale Dichte, IsolationForest zufällige Abtrennbarkeit und OneClassSVM eine flexible Grenze um typische Daten.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Semi-supervised Lernen mit wenigen Labels

    Teilen Sie `X_ssl_12`, `y_ssl_12` stratifiziert in Training und Test. Behalten Sie im Training nur drei bekannte Labels je Klasse und setzen Sie alle übrigen Trainingslabels auf `-1`.

1. Trainieren Sie `LabelPropagation` und `LabelSpreading`.
2. Bewerten Sie beide auf den vollständig gelabelten Testdaten.
3. Geben Sie vorhergesagte Klassenwahrscheinlichkeiten beziehungsweise Labelverteilungen für fünf ursprünglich ungelabelte Trainingspunkte aus.
4. Variieren Sie `gamma` oder `alpha` einmal und dokumentieren Sie die Empfindlichkeit.

> **Hinweis:** Die wahren Labels der maskierten Trainingspunkte dürfen nur zur nachträglichen Bewertung dienen.

In [ ]:
X_train_ssl, X_test_ssl, y_train_ssl, y_test_ssl = train_test_split(
    X_ssl_12, y_ssl_12, test_size=0.30,
    random_state=RANDOM_SEED, stratify=y_ssl_12
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Semi-supervised Lernen mit wenigen Labels
#
# Ziel dieser Codezelle:
# Teilen Sie Xssl12, yssl12 stratifiziert in Training und Test. Behalten Sie im
# Training nur drei bekannte Labels je Klasse und setzen Sie alle übrigen
# Trainingslabels auf -1. 1. Trainieren Sie LabelPropagation und Labe...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_train_ssl, X_test_ssl, y_train_ssl, y_test_ssl = train_test_split(
    X_ssl_12,
    y_ssl_12,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_ssl_12,
)

semi_labels = np.full_like(y_train_ssl, fill_value=-1)
labeled_indices = []
for class_value in np.unique(y_train_ssl):
    class_indices = np.flatnonzero(y_train_ssl == class_value)
    chosen = np.random.default_rng(RANDOM_SEED + int(class_value)).choice(
        class_indices,
        size=3,
        replace=False,
    )
    semi_labels[chosen] = class_value
    labeled_indices.extend(chosen.tolist())

propagation = LabelPropagation(kernel="rbf", gamma=20, max_iter=1000)
spreading = LabelSpreading(kernel="rbf", gamma=20, alpha=0.2, max_iter=1000)
propagation.fit(X_train_ssl, semi_labels)
spreading.fit(X_train_ssl, semi_labels)

propagation_accuracy = accuracy_score(
    y_test_ssl,
    propagation.predict(X_test_ssl),
)
spreading_accuracy = accuracy_score(
    y_test_ssl,
    spreading.predict(X_test_ssl),
)

unlabeled_indices = np.flatnonzero(semi_labels == -1)[:5]
distribution_table = pd.DataFrame(
    {
        "train_index": unlabeled_indices,
        "true_label_for_evaluation_only": y_train_ssl[unlabeled_indices],
        "propagation_p0": propagation.label_distributions_[unlabeled_indices, 0],
        "propagation_p1": propagation.label_distributions_[unlabeled_indices, 1],
        "spreading_p0": spreading.label_distributions_[unlabeled_indices, 0],
        "spreading_p1": spreading.label_distributions_[unlabeled_indices, 1],
    }
)

alternative_spreading = LabelSpreading(
    kernel="rbf",
    gamma=8,
    alpha=0.4,
    max_iter=1000,
)
alternative_spreading.fit(X_train_ssl, semi_labels)
alternative_accuracy = accuracy_score(
    y_test_ssl,
    alternative_spreading.predict(X_test_ssl),
)

print("Bekannte Trainingslabels:", len(labeled_indices), "von", len(y_train_ssl))
print("LabelPropagation-Testgenauigkeit:", round(propagation_accuracy, 3))
print("LabelSpreading-Testgenauigkeit:", round(spreading_accuracy, 3))
print("Alternative LabelSpreading-Testgenauigkeit:", round(alternative_accuracy, 3))
print("\nLabelverteilungen:")
print(distribution_table.round(3).to_string(index=False))

### Reflexion zu Aufgabe 3

Semi-supervised Verfahren nutzen die Geometrie vieler ungelabelter Punkte und wenige bestätigte Labels. Sie funktionieren nur gut, wenn nahe Punkte tatsächlich häufig dieselbe Klasse besitzen und die wenigen Labels repräsentativ sind. Die sichtbare Empfindlichkeit gegenüber `gamma` und `alpha` sollte als Unsicherheit dokumentiert werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Text bereinigen und drei Vektorisierungen untersuchen

    Bereinigen Sie `text_data_12["text"]` mit einer kleinen Funktion, die Kleinschreibung, Leerzeichenbereinigung und das Entfernen nicht alphabetischer Zeichen durchführt.

Vergleichen Sie:

- `CountVectorizer` mit Uni- und Bigrammen,
- `TfidfVectorizer` mit Uni- und Bigrammen,
- `HashingVectorizer(n_features=32, alternate_sign=False)`.

Geben Sie Matrixformen, Anzahl Nichtnullwerte, Vokabularbeispiele und die fünf höchsten TF-IDF-Gewichte des ersten Dokuments aus.

> **Hinweis:** Rufen Sie bei großen Textdaten nicht unkritisch `.toarray()` auf die gesamte Matrix auf.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Text bereinigen und drei Vektorisierungen untersuchen
#
# Ziel dieser Codezelle:
# Bereinigen Sie textdata12["text"] mit einer kleinen Funktion, die Kleinschreibung,
# Leerzeichenbereinigung und das Entfernen nicht alphabetischer Zeichen durchführt.
# Vergleichen Sie: - CountVectorizer mit Uni- und Bigr...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

def clean_german_text(text: str) -> str:
    # casefold vereinheitlicht Groß- und Kleinschreibung einschließlich
    # sprachspezifischer Zeichen. Ziffern und Satzzeichen werden entfernt.
    text = text.casefold()
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned_texts = text_data_12["text"].map(clean_german_text)

count_vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=1)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
hashing_vectorizer = HashingVectorizer(
    n_features=32,
    alternate_sign=False,
    norm=None,
    ngram_range=(1, 2),
)

count_matrix = count_vectorizer.fit_transform(cleaned_texts)
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_texts)
hashing_matrix = hashing_vectorizer.transform(cleaned_texts)

feature_names = tfidf_vectorizer.get_feature_names_out()
first_tfidf_values = tfidf_matrix[0].toarray().ravel()
top_indices = np.argsort(first_tfidf_values)[-5:][::-1]
top_terms = pd.DataFrame(
    {
        "term": feature_names[top_indices],
        "tfidf": first_tfidf_values[top_indices],
    }
)

print("Count:", count_matrix.shape, "Nichtnull:", count_matrix.nnz)
print("TF-IDF:", tfidf_matrix.shape, "Nichtnull:", tfidf_matrix.nnz)
print("Hashing:", hashing_matrix.shape, "Nichtnull:", hashing_matrix.nnz)
print("Vokabularbeispiele:", count_vectorizer.get_feature_names_out()[:15].tolist())
print("\nTop TF-IDF im ersten Dokument:")
print(top_terms.round(3).to_string(index=False))

### Reflexion zu Aufgabe 4

Count-Merkmale speichern Häufigkeiten, TF-IDF reduziert das Gewicht sehr häufiger corpusweiter Terme und Hashing benötigt kein gespeichertes Vokabular. Hashing kann Kollisionen erzeugen und Merkmalsnamen nicht direkt zurückgeben. Sparse-Matrizen vermeiden, dass die vielen Nullwerte als dichte Matrix unnötig Speicher belegen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integrationsaufgabe: Textpipelines vergleichen und speichern

    Teilen Sie die lokale Textsammlung stratifiziert in Training und Test. Vergleichen Sie:

1. CountVectorizer + MultinomialNB,
2. TfidfVectorizer + LogisticRegression.

Berechnen Sie Genauigkeit und Klassifikationsbericht, erstellen Sie eine Fehleranalyse mit Originaltext, wahrer und vorhergesagter Klasse und speichern/laden Sie die bessere Pipeline über einen `BytesIO`-Puffer mit `joblib`. Prüfen Sie, dass Vorhersagen vor und nach dem Laden identisch sind.

> **Hinweis:** Serialisieren Sie Vektorisierer und Modell gemeinsam als Pipeline.

In [ ]:
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    text_data_12["text"], text_data_12["label"], test_size=0.33,
    random_state=RANDOM_SEED, stratify=text_data_12["label"]
)

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integrationsaufgabe: Textpipelines vergleichen und speichern
#
# Ziel dieser Codezelle:
# Teilen Sie die lokale Textsammlung stratifiziert in Training und Test. Vergleichen
# Sie: 1. CountVectorizer + MultinomialNB, 2. TfidfVectorizer + LogisticRegression.
# Berechnen Sie Genauigkeit und Klassifikationsbericht...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    text_data_12["text"],
    text_data_12["label"],
    test_size=0.33,
    random_state=RANDOM_SEED,
    stratify=text_data_12["label"],
)

text_models = {
    "Count + NB": Pipeline(
        [
            ("vectorizer", CountVectorizer(ngram_range=(1, 2))),
            ("model", MultinomialNB(alpha=0.7)),
        ]
    ),
    "TF-IDF + LogReg": Pipeline(
        [
            ("vectorizer", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
            ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
        ]
    ),
}

model_rows = []
fitted_text_models = {}
for name, model in text_models.items():
    model.fit(X_text_train, y_text_train)
    predictions = model.predict(X_text_test)
    fitted_text_models[name] = model
    model_rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_text_test, predictions),
        }
    )
    print(f"\n{name}")
    print(classification_report(y_text_test, predictions, zero_division=0))

text_comparison = pd.DataFrame(model_rows).sort_values(
    "accuracy",
    ascending=False,
)
best_text_name = text_comparison.iloc[0]["model"]
best_text_model = fitted_text_models[best_text_name]
best_predictions = best_text_model.predict(X_text_test)

error_mask = best_predictions != y_text_test.to_numpy()
error_analysis = pd.DataFrame(
    {
        "text": X_text_test.to_numpy()[error_mask],
        "true_label": y_text_test.to_numpy()[error_mask],
        "predicted_label": best_predictions[error_mask],
    }
)

# BytesIO hält das Modell im Arbeitsspeicher und vermeidet lokale Pfade.
model_buffer = io.BytesIO()
joblib.dump(best_text_model, model_buffer)
model_buffer.seek(0)
loaded_text_model = joblib.load(model_buffer)
loaded_predictions = loaded_text_model.predict(X_text_test)

assert np.array_equal(best_predictions, loaded_predictions)

print("Vergleich:")
print(text_comparison.round(3).to_string(index=False))
print("\nBesseres Modell:", best_text_name)
print("\nFehleranalyse:")
print(error_analysis.to_string(index=False) if len(error_analysis) else "Keine Fehler in diesem kleinen Split.")

### Reflexion zu Aufgabe 5

Bei nur 24 Texten ist jede Testbewertung stark vom Split abhängig. Das Speichern der vollständigen Pipeline ist wichtig, weil Vokabular, IDF-Gewichte und Klassifikator gemeinsam benötigt werden. Eine fehlerfreie Vorhersage nach dem Laden prüft technische Reproduzierbarkeit, nicht die allgemeine Qualität auf neuen Sprachvarianten oder Themen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.